# 리뷰 데이터 DB 추출 및 전처리

Steam API로 수집된 리뷰 데이터를 DB에서 추출하여 `data/raw/`에 저장하고,
전처리하여 `data/preprocessed/`에 저장한다.

> **주의:** 이 노트북은 아래 수집 스크립트 실행 완료 후 실행해야 한다.
> ```
> python src/collect/jin/collect_steam_indie_stratified_reviews.py \
>     data/preprocessed/steam_indie_genre_stratified_sample.csv
> python src/collect/jin/collect_missing_review_summary.py
> python src/collect/jin/collect_steam_indie_stratified_histogram.py \
>     data/preprocessed/steam_indie_genre_stratified_sample.csv
> ```

## DB 추출 대상

| 테이블 | 설명 | 저장 파일 |
|---|---|---|
| `steam_indie_reviews` | 리뷰 원문 및 작성자 정보 | `data/raw/steam_indie_reviews.csv` |
| `steam_indie_review_summary` | 게임별 리뷰 요약 통계 | `data/raw/steam_indie_review_summary.csv` |
| `steam_indie_review_histogram` | 월별/일별 리뷰 집계 | `data/raw/steam_indie_review_histogram.csv` |

## 전처리 결과

| 대상 | 설명 | 저장 파일 |
|---|---|---|
| `steam_indie_review_summary` | `review_score_desc` 소문자 변환 | `data/preprocessed/steam_indie_review_summary.csv` |
| `steam_indie_review_histogram` | 파생 컬럼 추가, 분석 대상 필터링 | `data/preprocessed/steam_indie_review_histogram.csv` |
| `steam_indie_reviews` | 결측 제거, 날짜 변환, 플레이타임 정제 | `data/preprocessed/steam_indie_reviews.csv` |

## 라이브러리 임포트 및 DB 연결

In [1]:
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve().parents[1] / 'src'))

from utils.db import get_engine

warnings.filterwarnings('ignore')

RAW_DIR          = Path('..').resolve().parents[1] / 'data' / 'raw'
PREPROCESSED_DIR = Path('..').resolve().parents[1] / 'data' / 'preprocessed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

conn = get_engine()
print('DB 연결 성공')
print(f'RAW_DIR: {RAW_DIR}')
print(f'PREPROCESSED_DIR: {PREPROCESSED_DIR}')

DB 연결 성공
RAW_DIR: /Users/jin/Develop/codingclub/game-analysis/data/raw
PREPROCESSED_DIR: /Users/jin/Develop/codingclub/game-analysis/data/preprocessed


## 1. steam_indie_reviews

수집된 전체 리뷰 데이터. 용량이 크므로 청크 단위로 읽어 저장한다.

In [2]:
out = RAW_DIR / 'steam_indie_reviews.csv'

chunk_size = 50000
total = 0
for i, chunk in enumerate(
    pd.read_sql('SELECT * FROM steam_indie_reviews ORDER BY appid, timestamp_created', conn, chunksize=chunk_size)
):
    chunk.to_csv(out, index=False, encoding='utf-8-sig', mode='w' if i == 0 else 'a', header=(i == 0))
    total += len(chunk)
    print(f'  청크 {i+1}: {total:,}행 저장 완료')

print(f'\nsteam_indie_reviews: 총 {total:,}행 → {out.name}')

  청크 1: 50,000행 저장 완료
  청크 2: 100,000행 저장 완료
  청크 3: 150,000행 저장 완료
  청크 4: 166,766행 저장 완료

steam_indie_reviews: 총 166,766행 → steam_indie_reviews.csv


## 2. steam_indie_review_summary

게임별 리뷰 요약 통계 (review_score, 긍정/부정 수 등). 수집 시점의 전체 누적 리뷰 기준이다.

In [3]:
df_summary = pd.read_sql('SELECT * FROM steam_indie_review_summary ORDER BY appid', conn)

out = RAW_DIR / 'steam_indie_review_summary.csv'
df_summary.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_summary: {len(df_summary):,}행 → {out.name}')
print(f'컬럼: {df_summary.columns.tolist()}')
df_summary.head(3)

steam_indie_review_summary: 200행 → steam_indie_review_summary.csv
컬럼: ['appid', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews']


,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews
0,324470,6,Mostly Positive,118,33,151
1,571740,8,Very Positive,22056,2260,24316
2,588440,8,Very Positive,64,14,78


## 3. steam_indie_review_histogram

게임별 월별(`rollups`) 및 일별(`recent`) 리뷰 집계 데이터.

In [4]:
df_hist = pd.read_sql(
    'SELECT * FROM steam_indie_review_histogram ORDER BY appid, data_type, date', conn
)

out = RAW_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_histogram: {len(df_hist):,}행 → {out.name}')
print(f'컬럼: {df_hist.columns.tolist()}')
print(f'\ndata_type 분포:')
print(df_hist['data_type'].value_counts().to_string())
df_hist.head(3)

steam_indie_review_histogram: 11,615행 → steam_indie_review_histogram.csv
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']

data_type 분포:
data_type
recent     5947
rollups    5668


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,324470,SinaRun,Racing_mid,2025-11-03,2015-10-26,2026-04-26,2026-03-28,0,0,recent
1,324470,SinaRun,Racing_mid,2025-11-03,2015-10-26,2026-04-26,2026-03-29,0,0,recent
2,324470,SinaRun,Racing_mid,2025-11-03,2015-10-26,2026-04-26,2026-03-30,0,0,recent


## 4. DB 연결 종료 및 저장 결과 요약

In [5]:
conn.dispose()
print('DB 연결 종료')

print('\n=== 저장 완료 파일 목록 ===')
for f in sorted(RAW_DIR.glob('steam_indie_review*.csv')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<50} {size_mb:>7.1f} MB')

DB 연결 종료

=== 저장 완료 파일 목록 ===
  steam_indie_review_histogram.csv                       1.0 MB
  steam_indie_review_summary.csv                         0.0 MB
  steam_indie_reviews.csv                               60.2 MB


---

## steam_indie_review_summary 전처리

- `review_score_desc`: 소문자 변환
- → `data/preprocessed/steam_indie_review_summary.csv`

In [ ]:
df_summary['review_score_desc'] = df_summary['review_score_desc'].str.lower()

out_path = PREPROCESSED_DIR / 'steam_indie_review_summary.csv'
df_summary.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_summary):,}개)')
print(df_summary['review_score_desc'].value_counts().to_string())

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_review_summary.csv (200개)
review_score_desc
very positive              65
positive                   49
mixed                      32
mostly positive            24
overwhelmingly positive     8
mostly negative             4
no user reviews             4
4 user reviews              3
6 user reviews              3
7 user reviews              2
9 user reviews              2
8 user reviews              1
5 user reviews              1
3 user reviews              1
negative                    1
(200, 6)


---

## steam_indie_review_histogram 전처리

| 처리 항목 | 결과 |
|---|---|
| 문자열 공백 | 문자열 컬럼 앞뒤 공백 제거, `data_type` 소문자 통일 |
| 날짜 변환 | `release_date`, `hist_start_date`, `hist_end_date`, `date` → datetime |
| 리뷰 수 합계 | `recommendations_total` 생성 (`recommendations_up + recommendations_down`) |
| 분석 대상 필터링 | `steam_indie_games` 기준 inner join (분석 대상 게임의 히스토그램만 유지) |

In [7]:
df_hist = pd.read_csv(RAW_DIR / 'steam_indie_review_histogram.csv')
print(f'로드 완료: {df_hist.shape}')
print(f'컬럼: {df_hist.columns.tolist()}')

로드 완료: (11615, 10)
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']


In [8]:
# 문자열 공백 정리
string_cols = df_hist.select_dtypes(include=['object', 'string']).columns.tolist()
for col in string_cols:
    df_hist[col] = df_hist[col].astype('string').str.strip()
if 'name' in df_hist.columns:
    df_hist['name'] = df_hist['name'].str.replace(r'\s+', ' ', regex=True)
df_hist['data_type'] = df_hist['data_type'].str.lower()

# 날짜형 변환
for col in ['release_date', 'hist_start_date', 'hist_end_date', 'date']:
    df_hist[col] = pd.to_datetime(df_hist[col], errors='coerce')

# 리뷰 수 합계
for col in ['appid', 'recommendations_up', 'recommendations_down']:
    df_hist[col] = pd.to_numeric(df_hist[col], errors='coerce')
df_hist['recommendations_total'] = df_hist['recommendations_up'] + df_hist['recommendations_down']

# steam_indie_games 기준 inner join (분석 대상 게임의 히스토그램만 유지)
games_appids = pd.read_csv(PREPROCESSED_DIR / 'steam_indie_games.csv', usecols=['appid'])
before = len(df_hist)
df_hist = df_hist.merge(games_appids, on='appid', how='inner')
print(f'inner join 결과: {before:,} → {len(df_hist):,}행 ({before - len(df_hist):,}개 제외)')

print('전처리 완료')
print(f'shape: {df_hist.shape}')
print(f'컬럼: {df_hist.columns.tolist()}')

inner join 결과: 11,615 → 11,615행 (0개 제외)
전처리 완료
shape: (11615, 11)
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type', 'recommendations_total']


In [9]:
out_path = PREPROCESSED_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_hist):,}개)')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_review_histogram.csv (11,615개)


---

## steam_indie_reviews 전처리

| 처리 항목 | 결과 |
|---|---|
| 결측 제거 | `review` 컬럼 결측 행 제거 |
| 분석 대상 필터링 | `steam_indie_games` 기준 inner join (분석 대상 게임 리뷰만 유지) |
| 날짜 변환 | `timestamp_created` → `created_date`, `timestamp_updated` → `updated_date`, `author_last_played` → `author_last_played_date` |
| 플레이타임 | `author_playtime_*` 3개 컬럼 분 → 시간 단위 변환 후 원본 제거 |

In [10]:
reviews = pd.read_csv(RAW_DIR / 'steam_indie_reviews.csv')
print(f'로드 완료: {reviews.shape}')

# 결측 제거
reviews_clean = reviews.copy()
reviews_clean = reviews_clean.dropna(subset=['review'])

# steam_indie_games 기준 inner join (분석 대상 게임의 리뷰만 유지)
games_appids = pd.read_csv(PREPROCESSED_DIR / 'steam_indie_games.csv', usecols=['appid'])
before = len(reviews_clean)
reviews_clean = reviews_clean.merge(games_appids, on='appid', how='inner')
print(f'inner join 결과: {before:,} → {len(reviews_clean):,}행 ({before - len(reviews_clean):,}개 제외)')

# 날짜 변환 (Unix timestamp → datetime)
for col in ['timestamp_created', 'timestamp_updated', 'author_last_played']:
    new_col = col.replace('timestamp_', '') + '_date' if col.startswith('timestamp_') else 'author_last_played_date'
    reviews_clean[new_col] = pd.to_datetime(reviews_clean[col], unit='s', errors='coerce')

# 플레이타임 분 → 시간 변환 후 원본 제거
playtime_cols = ['author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review']
for col in playtime_cols:
    new_col = col.replace('author_playtime_', 'playtime_') + '_hours'
    reviews_clean[new_col] = reviews_clean[col] / 60
reviews_clean = reviews_clean.drop(columns=playtime_cols)

print(f'전처리 완료: {reviews_clean.shape}')
print(f'컬럼: {reviews_clean.columns.tolist()}')

로드 완료: (166766, 21)
inner join 결과: 166,385 → 166,385행 (0개 제외)
전처리 완료: (166385, 24)
컬럼: ['recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_last_played', 'created_date', 'updated_date', 'author_last_played_date', 'playtime_forever_hours', 'playtime_last_two_weeks_hours', 'playtime_at_review_hours']


In [11]:
out_path = PREPROCESSED_DIR / 'steam_indie_reviews.csv'
reviews_clean.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(reviews_clean):,}개)')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_reviews.csv (166,385개)
